In [1]:
%load_ext autoreload
%autoreload 2

In [8]:
import random
import numpy as np
from src.river_sddm import RiverSDDM

def test_sddm_local_drift(incremental = False):
    # 1. Konfiguracja detektora dla klastra
    features = ['sensor_a', 'sensor_b', 'sensor_c']
    detector = RiverSDDM(
        n_bins=10, 
        ref_window_size=200,  # Rozmiar okna referencyjnego [cite: 415]
        cur_window_size=100,  # Rozmiar bieżącego okna (batch p) [cite: 292]
        threshold=0.6,        # Próg dla nagłego dryfu (abrupt drift) [cite: 578]
        test_interval=5,      # Częstotliwość testowania [cite: 243]
        alpha=1.0,
        incremental= incremental                          
    )

    print(f"Features: {features}\n")

    # 2. Generowanie strumienia danych
    for t in range(1, 5000):
        # Stan normalny: wszystkie cechy w zakresie [0.0, 0.4]
        if t < 800:
            x = {f: random.uniform(0.0, 0.4) for f in features}
        
        # Symulacja dryfu: tylko 'sensor_b' zmienia zakres na [0.6, 1.0]
        elif t >=800 and t < 2000: 
            x = {
                'sensor_a': random.uniform(0.0, 0.4),
                'sensor_b': random.uniform(0.6, 0.8), # Tu następuje dryf [cite: 104]
                'sensor_c': random.uniform(0.0, 0.4)
            }
        else:
            x = {
                'sensor_a': random.uniform(0.0, 0.4),
                'sensor_b': random.uniform(0.8, 1), # Tu następuje dryf [cite: 104]
                'sensor_c': random.uniform(0.0, 0.4)
            }
        # Aktualizacja detektora (obsługuje x jako dict i opcjonalne y) [cite: 18, 80]
        detector.update(x, y=None) # Test bez etykiet (P(X))

        # 3. Sprawdzanie detekcji
        if detector.drift_detected:
            report = detector.get_drift_report()
            print(f"ALARM: Local drift detected!")
            print(f"  - Time step: {t}")
            print(f"  - Drift magnitude: {report['magnitude']:.4f}")
            print(f"  - Drift source: {report['source']}")
            print(f"  - Window status: Resetting approach")
            print("-" * 40)


In [9]:
test_sddm_local_drift(False)

Features: ['sensor_a', 'sensor_b', 'sensor_c']

ALARM: Local drift detected!
  - Time step: 865
  - Drift magnitude: 0.7216
  - Drift source: sensor_b
  - Window status: Resetting approach
----------------------------------------
ALARM: Local drift detected!
  - Time step: 2060
  - Drift magnitude: 0.6377
  - Drift source: sensor_b
  - Window status: Resetting approach
----------------------------------------


In [10]:
test_sddm_local_drift(True)

Features: ['sensor_a', 'sensor_b', 'sensor_c']

ALARM: Local drift detected!
  - Time step: 855
  - Drift magnitude: 0.6561
  - Drift source: sensor_b
  - Window status: Resetting approach
----------------------------------------
ALARM: Local drift detected!
  - Time step: 2060
  - Drift magnitude: 0.6884
  - Drift source: sensor_b
  - Window status: Resetting approach
----------------------------------------
